# Module 7: Deep Dive into LLMs

LLMs are transformer models scaled to billions of parameters and trained on internet-scale text. This module covers the practical skills for working with them: prompt engineering, fine-tuning strategies (SFT, DPO, RLHF), RAG, and evaluation.

## 1. Prompt Engineering Patterns

Prompt engineering is the art of communicating with an LLM. The difference between a good and bad prompt can be the difference between 60% and 95% accuracy on a task — without changing the model at all.

**The five patterns every ML engineer should know:**

In [ ]:
PATTERNS = {
    "zero_shot": {
        "desc": "Direct instruction, no examples. Works for simple tasks.",
        "example": "Classify this ticket: 'I was charged twice' -> Category:",
        "when": "Task is simple, model is capable, you want speed",
    },
    "few_shot": {
        "desc": "Provide 2-5 examples to establish the pattern.",
        "example": "Ticket: 'charged twice' -> billing | Ticket: 'app crashes' -> technical",
        "when": "Model needs to understand format/style, task is moderately complex",
    },
    "chain_of_thought": {
        "desc": "Ask model to reason step-by-step before answering.",
        "example": "Step 1: Identify the issue. Step 2: Determine department. Step 3: Classify.",
        "when": "Complex reasoning, math, multi-step logic",
    },
    "role_based": {
        "desc": "Assign the model a persona/expertise.",
        "example": "You are a senior support manager with 10 years experience...",
        "when": "You want domain-specific framing, or more careful/structured output",
    },
    "structured_output": {
        "desc": "Force JSON/structured responses for downstream parsing.",
        "example": "Return ONLY valid JSON: {category, confidence, reasoning}",
        "when": "Output feeds into another system, you need parseable format",
    },
}

for name, info in PATTERNS.items():
    print("=" * 50)
    print(name.upper())
    print("=" * 50)
    print(f"  What: {info['desc']}")
    print(f"  When: {info['when']}")
    print(f"  Example: {info['example']}")
    print()

## 2. Fine-Tuning Strategies

Pre-trained LLMs know language but don't know YOUR task. Fine-tuning adapts them:

| Strategy | What It Does | When to Use |
|----------|-------------|------------|
| **SFT (Supervised Fine-Tuning)** | Train on (input, desired_output) pairs | You have labeled examples of the task |
| **DPO (Direct Preference Optimization)** | Train on (input, preferred, rejected) triples | You have human preferences between outputs |
| **RLHF** | Train reward model from preferences, then RL | Large scale, complex alignment |
| **LoRA/PEFT** | Train only small adapter weights, freeze base model | Limited GPU memory, fast iteration |

**Why LoRA?** A 7B parameter model requires ~28GB in float32. Fine-tuning all parameters needs 3-4x that for gradients/optimizer states. LoRA adds trainable low-rank matrices (0.1-1% of parameters) to each layer, achieving 90-95% of full fine-tuning quality with a fraction of the compute.

In [ ]:
lora_config = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
}

base_params = 7_000_000_000
lora_params_per_layer = 2 * 4096 * 16 * 4
num_layers = 32
total_lora_params = lora_params_per_layer * num_layers

print(f"Base model parameters: {base_params:,}")
print(f"LoRA parameters:       {total_lora_params:,}")
print(f"Trainable ratio:       {total_lora_params/base_params:.2%}")
print(f"\nThis means we only train {total_lora_params/base_params:.2%} of the model")
print(f"but get ~90-95% of full fine-tuning quality.")

## 3. RAG (Retrieval-Augmented Generation)

**The problem:** LLMs hallucinate. They generate plausible-sounding but factually wrong answers because they rely solely on training data (which may be outdated or incomplete).

**The solution:** Before generating an answer, retrieve relevant documents from a knowledge base and include them in the prompt. The LLM generates grounded in real evidence.

**Pipeline:** Query -> Embed -> Retrieve (FAISS/vector DB) -> Re-rank -> Context Assembly -> LLM -> Answer

In [ ]:
import numpy as np

class SimpleRAG:
    def __init__(self, dim=64):
        self.dim = dim
        self.docs = []
        self.vecs = []

    def _embed(self, text):
        np.random.seed(abs(hash(text)) % 2**31)
        v = np.random.randn(self.dim).astype(np.float32)
        return v / np.linalg.norm(v)

    def index(self, documents):
        self.docs = documents
        self.vecs = [self._embed(d) for d in documents]
        print(f"Indexed {len(documents)} documents")

    def query(self, question, top_k=3):
        q_vec = self._embed(question)
        scores = [float(np.dot(q_vec, v)) for v in self.vecs]
        top_idx = np.argsort(scores)[-top_k:][::-1]

        retrieved = [(self.docs[i], scores[i]) for i in top_idx]
        return retrieved

rag = SimpleRAG()
rag.index([
    "RAG combines retrieval with generation to ground responses in facts.",
    "FAISS enables fast similarity search on dense vector embeddings.",
    "Cross-encoder re-ranking improves precision by scoring query-doc pairs.",
    "RAGAS evaluates RAG on faithfulness, answer relevance, context precision.",
    "Hybrid search combines dense embeddings with sparse BM25 for better recall.",
])

results = rag.query("How do you evaluate a RAG pipeline?")
print(f"\nQuestion: How do you evaluate a RAG pipeline?\n")
print("Retrieved documents:")
for doc, score in results:
    print(f"  [{score:.3f}] {doc}")

## Key Takeaways

1. **Prompt engineering** is the fastest way to improve LLM performance — no training needed
2. **LoRA** makes fine-tuning accessible on consumer GPUs (0.1% parameters, 90%+ quality)
3. **DPO > RLHF** for most practical fine-tuning — simpler, more stable, no reward model needed
4. **RAG** is essential for production LLMs — it grounds responses in facts and reduces hallucination
5. **Always evaluate** with RAGAS/DeepEval — gut feeling is not a metric